In [ ]:
import sys, subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-U",
    "transformers", "datasets", "accelerate", "torch", "scikit-learn"
])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 12.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.4 MB/s  0:00:000m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 13.8 MB/s  0:01:02m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 13.8 MB/s  0:00:41m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 13.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 16.4 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 40.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 14.0 MB/s  0:00:13m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 15.6 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 13.8 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.23.0+cu129 requires torch==2.8.0, but you have torch 2.9.1 which is incompatible.
torchaudio 2.8.0+cu129 requires torch==2.8.0, but you have torch 2.9.1 which is incompatible.


CompletedProcess(args=['/venv/main/bin/python', '-m', 'pip', 'install', '-U', 'transformers', 'datasets', 'accelerate', 'torch', 'scikit-learn'], returncode=0)

In [ ]:
import torch, transformers, accelerate
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("cuda disponível:", torch.cuda.is_available())

torch: 2.9.1+cu128
transformers: 4.57.3
accelerate: 1.12.0
cuda disponível: True


In [ ]:
"""
=============================================================================
VALIDAÇÃO COMPLETA: BERT-Tiny-PT (Destilação Total vs Parcial)
=============================================================================

Modelos avaliados:
1. BERT-Tiny-PT Total: Pedro-Rebollo20/BERT-Tiny-Port
2. BERT-Tiny-PT Parcial: Pedro-Rebollo20/Bert_tiny_pt_parcial
3. BERT-Tiny Original (EN): prajjwal1/bert-tiny
4. BERTimbau-base (Teacher): neuralmind/bert-base-portuguese-cased

Tarefas de avaliação:
- MLM Loss (Masked Language Modeling)
- Sentiment Analysis (TweetSentBR)
- Textual Similarity (ASSIN2 - STS)
- Textual Entailment (ASSIN2 - RTE)
- Análise de eficiência (parâmetros, velocidade)
=============================================================================
"""

import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset
from torch.utils.data import DataLoader
from scipy.stats import pearsonr
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
import pandas as pd
import re
from typing import Dict, List
import time

# =============================================================================
# CONFIGURAÇÃO
# =============================================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}\n")

# Modelos a serem avaliados
MODELS = {
    "BERT-Tiny-PT (Total)": "Pedro-Rebollo20/BERT-Tiny-Port",
    "BERT-Tiny-PT (Parcial)": "Pedro-Rebollo20/Bert_tiny_pt_parcial",
    "BERT-Tiny (EN)": "prajjwal1/bert-tiny",
    "BERTimbau-base": "neuralmind/bert-base-portuguese-cased",
}

MAX_LENGTH = 128
SEED = 42

# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def clean_text(text: str) -> str:
    """Remove tags HTML e normaliza espaços."""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else "."


def count_parameters(model) -> int:
    """Conta parâmetros treináveis do modelo."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def get_model_size_mb(model_name: str) -> float:
    """Estima tamanho do modelo em MB."""
    model = AutoModel.from_pretrained(model_name)
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    size_mb = (param_size + buffer_size) / (1024 ** 2)
    del model
    torch.cuda.empty_cache()
    return size_mb


# =============================================================================
# 1. AVALIAÇÃO MLM (Masked Language Modeling)
# =============================================================================

def evaluate_mlm_loss(model, dataloader) -> float:
    """Calcula MLM loss média no conjunto de teste."""
    model.eval()
    total_loss, total_tokens = 0.0, 0

    with torch.no_grad():
        for batch in dataloader:
            batch = {
                k: v.to(model.device) if isinstance(v, torch.Tensor) else v
                for k, v in batch.items()
            }
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            mask = batch["labels"] != -100
            num_masked = mask.sum().item()

            if num_masked > 0:
                total_loss += outputs.loss.item() * num_masked
                total_tokens += num_masked

    return total_loss / total_tokens if total_tokens > 0 else float("nan")


def build_mlm_loader(ds, tokenizer, batch_size=8):
    """Cria DataLoader para avaliação MLM."""
    def tokenize_fn(batch):
        textos = [clean_text(t) for t in batch["text"]]
        return tokenizer(
            textos,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    ds_tok = ds.map(tokenize_fn, batched=True)
    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm_probability=0.15
    )

    loader = DataLoader(
        ds_tok.remove_columns(
            [c for c in ds_tok.column_names if c not in ["input_ids", "attention_mask"]]
        ),
        batch_size=batch_size,
        collate_fn=collator,
    )
    return loader


def run_mlm_evaluation(models_dict: Dict[str, str], test_dataset) -> pd.DataFrame:
    """Executa avaliação MLM para todos os modelos."""
    print("=" * 80)
    print("AVALIAÇÃO 1: MLM LOSS (Masked Language Modeling)")
    print("=" * 80)

    results = []

    for model_name, model_path in models_dict.items():
        print(f"\nAvaliando {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForMaskedLM.from_pretrained(model_path).to(DEVICE)

        loader = build_mlm_loader(test_dataset, tokenizer)

        start_time = time.time()
        mlm_loss = evaluate_mlm_loss(model, loader)
        elapsed_time = time.time() - start_time

        results.append({
            "Modelo": model_name,
            "MLM Loss": mlm_loss,
            "Tempo (s)": elapsed_time,
        })

        print(f"  MLM Loss: {mlm_loss:.4f}")
        print(f"  Tempo: {elapsed_time:.2f}s")

        del model, tokenizer
        torch.cuda.empty_cache()

    df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print(df.to_string(index=False))
    print("=" * 80 + "\n")
    return df


# =============================================================================
# 2. AVALIAÇÃO SENTIMENT ANALYSIS (TweetSentBR)
# =============================================================================

def run_sentiment_evaluation(models_dict: Dict[str, str]) -> pd.DataFrame:
    """Executa avaliação de análise de sentimento."""
    print("=" * 80)
    print("AVALIAÇÃO 2: SENTIMENT ANALYSIS (TweetSentBR)")
    print("=" * 80)

    # Carrega dataset
    tweet = load_dataset("eduagarcia/tweetsentbr_fewshot")
    tweet = tweet["train"].train_test_split(test_size=0.2, seed=SEED)
    train_ds, test_ds = tweet["train"], tweet["test"]

    print(f"Treino: {len(train_ds)} | Teste: {len(test_ds)}\n")

    label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}
    id2label = {v: k for k, v in label2id.items()}

    def tokenize_fn(batch, tokenizer):
        return tokenizer(
            batch["sentence"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    def encode_labels(example):
        return {"labels": label2id[example["label"]]}

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="macro"),
        }

    results = []

    for model_name, model_path in models_dict.items():
        print(f"\nTreinando e avaliando {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_path)

        # Tokeniza
        train_tok = train_ds.map(lambda x: tokenize_fn(x, tokenizer), batched=True)
        test_tok = test_ds.map(lambda x: tokenize_fn(x, tokenizer), batched=True)

        train_tok = train_tok.map(encode_labels)
        test_tok = test_tok.map(encode_labels)

        cols_keep = ["input_ids", "attention_mask", "labels"]
        cols_remove = [c for c in train_tok.column_names if c not in cols_keep]

        train_tok = train_tok.remove_columns(cols_remove)
        test_tok = test_tok.remove_columns(cols_remove)

        # Modelo
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=3, id2label=id2label, label2id=label2id
        ).to(DEVICE)

        # Treino
        args = TrainingArguments(
            output_dir=f"./eval_sentiment_{model_name.replace(' ', '_')}",
            overwrite_output_dir=True,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            learning_rate=2e-5,
            num_train_epochs=3,
            weight_decay=0.01,
            logging_steps=100,
            save_total_limit=1,
            report_to="none",
            seed=SEED,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_tok,
            eval_dataset=test_tok,
            compute_metrics=compute_metrics,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        )

        start_time = time.time()
        trainer.train()
        metrics = trainer.evaluate()
        elapsed_time = time.time() - start_time

        results.append({
            "Modelo": model_name,
            "Accuracy": metrics["eval_accuracy"],
            "F1 (macro)": metrics["eval_f1"],
            "Tempo (s)": elapsed_time,
        })

        print(f"  Accuracy: {metrics['eval_accuracy']:.4f}")
        print(f"  F1: {metrics['eval_f1']:.4f}")

        del model, tokenizer, trainer
        torch.cuda.empty_cache()

    df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print(df.to_string(index=False))
    print("=" * 80 + "\n")
    return df


# =============================================================================
# 3. AVALIAÇÃO TEXTUAL SIMILARITY (ASSIN2 - STS)
# =============================================================================

def embed_sentence(model, tokenizer, sentence):
    """Gera embedding médio da sentença."""
    enc = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        out = model(**enc, output_hidden_states=False)

    emb = out.last_hidden_state.mean(dim=1)
    return emb.squeeze(0)


def evaluate_sts(model, tokenizer, dataset):
    """Avalia similaridade textual usando correlação de Pearson."""
    preds, gold = [], []

    model.eval()
    for item in dataset:
        s1 = item["premise"]
        s2 = item["hypothesis"]

        emb1 = embed_sentence(model, tokenizer, s1)
        emb2 = embed_sentence(model, tokenizer, s2)

        sim = F.cosine_similarity(emb1, emb2, dim=0).item()
        preds.append(sim)
        gold.append(item["relatedness_score"])

    pearson = pearsonr(preds, gold)[0]
    return pearson


def run_sts_evaluation(models_dict: Dict[str, str]) -> pd.DataFrame:
    """Executa avaliação de similaridade textual."""
    print("=" * 80)
    print("AVALIAÇÃO 3: TEXTUAL SIMILARITY (ASSIN2 - STS)")
    print("=" * 80)

    assin = load_dataset("assin2")
    test_assin = assin["test"]

    print(f"Conjunto de teste: {len(test_assin)} exemplos\n")

    results = []

    for model_name, model_path in models_dict.items():
        print(f"\nAvaliando {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        model = AutoModel.from_pretrained(model_path).to(DEVICE)

        start_time = time.time()
        pearson = evaluate_sts(model, tokenizer, test_assin)
        elapsed_time = time.time() - start_time

        results.append({
            "Modelo": model_name,
            "Pearson": pearson,
            "Tempo (s)": elapsed_time,
        })

        print(f"  Pearson: {pearson:.4f}")
        print(f"  Tempo: {elapsed_time:.2f}s")

        del model, tokenizer
        torch.cuda.empty_cache()

    df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print(df.to_string(index=False))
    print("=" * 80 + "\n")
    return df


# =============================================================================
# 4. AVALIAÇÃO TEXTUAL ENTAILMENT (ASSIN2 - RTE)
# =============================================================================

def run_rte_evaluation(models_dict: Dict[str, str]) -> pd.DataFrame:
    """Executa avaliação de inferência textual."""
    print("=" * 80)
    print("AVALIAÇÃO 4: TEXTUAL ENTAILMENT (ASSIN2 - RTE)")
    print("=" * 80)

    assin = load_dataset("assin2")
    train = assin["train"]
    test = assin["validation"]

    print(f"Treino: {len(train)} | Teste: {len(test)}\n")

    label2id = {"entailment": 0, "contradiction": 1, "neutral": 2}
    id2label = {v: k for k, v in label2id.items()}

    def tokenize_batch(batch, tokenizer):
        return tokenizer(
            batch["premise"],
            batch["hypothesis"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    def encode_labels(example):
        return {"labels": int(example["entailment_judgment"])}

    def compute_metrics(pred):
        logits, labels = pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="macro"),
        }

    results = []

    for model_name, model_path in models_dict.items():
        print(f"\nTreinando e avaliando {model_name}...")

        tokenizer = AutoTokenizer.from_pretrained(model_path)

        # Tokeniza
        train_tok = train.map(lambda x: tokenize_batch(x, tokenizer), batched=True)
        test_tok = test.map(lambda x: tokenize_batch(x, tokenizer), batched=True)

        train_tok = train_tok.map(encode_labels)
        test_tok = test_tok.map(encode_labels)

        cols_keep = ["input_ids", "attention_mask", "labels"]
        cols_remove = [c for c in train_tok.column_names if c not in cols_keep]

        train_tok = train_tok.remove_columns(cols_remove)
        test_tok = test_tok.remove_columns(cols_remove)

        # Modelo
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=3, id2label=id2label, label2id=label2id
        ).to(DEVICE)

        # Treino
        args = TrainingArguments(
            output_dir=f"./eval_rte_{model_name.replace(' ', '_')}",
            overwrite_output_dir=True,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            learning_rate=2e-5,
            num_train_epochs=3,
            weight_decay=0.01,
            logging_steps=100,
            save_total_limit=1,
            report_to="none",
            seed=SEED,
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_tok,
            eval_dataset=test_tok,
            compute_metrics=compute_metrics,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        )

        start_time = time.time()
        trainer.train()
        metrics = trainer.evaluate()
        elapsed_time = time.time() - start_time

        results.append({
            "Modelo": model_name,
            "Accuracy": metrics["eval_accuracy"],
            "F1 (macro)": metrics["eval_f1"],
            "Tempo (s)": elapsed_time,
        })

        print(f"  Accuracy: {metrics['eval_accuracy']:.4f}")
        print(f"  F1: {metrics['eval_f1']:.4f}")

        del model, tokenizer, trainer
        torch.cuda.empty_cache()

    df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print(df.to_string(index=False))
    print("=" * 80 + "\n")
    return df


# =============================================================================
# 5. ANÁLISE DE EFICIÊNCIA
# =============================================================================

def run_efficiency_analysis(models_dict: Dict[str, str]) -> pd.DataFrame:
    """Analisa eficiência dos modelos."""
    print("=" * 80)
    print("ANÁLISE DE EFICIÊNCIA")
    print("=" * 80)

    results = []

    for model_name, model_path in models_dict.items():
        print(f"\nAnalisando {model_name}...")

        model = AutoModel.from_pretrained(model_path)
        n_params = count_parameters(model)
        size_mb = get_model_size_mb(model_path)

        results.append({
            "Modelo": model_name,
            "Parâmetros (M)": n_params / 1e6,
            "Tamanho (MB)": size_mb,
        })

        print(f"  Parâmetros: {n_params / 1e6:.2f}M")
        print(f"  Tamanho: {size_mb:.2f} MB")

        del model
        torch.cuda.empty_cache()

    df = pd.DataFrame(results)
    print("\n" + "=" * 80)
    print(df.to_string(index=False))
    print("=" * 80 + "\n")
    return df


# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================

if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("VALIDAÇÃO COMPLETA: BERT-Tiny-PT")
    print("=" * 80 + "\n")

    # Carrega dataset brWaC para MLM
    print("Carregando dataset brWaC...")
    ds_brwac = load_dataset("nlpufg/brwac", split="train")
    ds_split = ds_brwac.train_test_split(test_size=0.1, seed=SEED)
    ds_test = ds_split["test"]
    print(f"Dataset de teste: {len(ds_test)} exemplos\n")

    # Dicionário de resultados
    all_results = {}

    # 1. MLM Loss
    all_results["mlm"] = run_mlm_evaluation(MODELS, ds_test)

    # 2. Sentiment Analysis
    all_results["sentiment"] = run_sentiment_evaluation(MODELS)

    # 3. Textual Similarity
    all_results["sts"] = run_sts_evaluation(MODELS)

    # 4. Textual Entailment
    all_results["rte"] = run_rte_evaluation(MODELS)

    # 5. Eficiência
    all_results["efficiency"] = run_efficiency_analysis(MODELS)

    # =============================================================================
    # RESUMO FINAL
    # =============================================================================

    print("\n" + "=" * 80)
    print("RESUMO FINAL - COMPARAÇÃO ENTRE MODELOS")
    print("=" * 80 + "\n")

    # Cria tabela consolidada
    summary = pd.DataFrame({
        "Modelo": all_results["mlm"]["Modelo"],
        "MLM Loss": all_results["mlm"]["MLM Loss"],
        "Sentiment F1": all_results["sentiment"]["F1 (macro)"],
        "STS Pearson": all_results["sts"]["Pearson"],
        "RTE F1": all_results["rte"]["F1 (macro)"],
        "Parâmetros (M)": all_results["efficiency"]["Parâmetros (M)"],
        "Tamanho (MB)": all_results["efficiency"]["Tamanho (MB)"],
    })

    print(summary.to_string(index=False))

    # Análise comparativa
    print("\n" + "=" * 80)
    print("ANÁLISE COMPARATIVA")
    print("=" * 80 + "\n")

    # Compara Tiny-PT Total vs Parcial
    total_idx = summary[summary["Modelo"] == "BERT-Tiny-PT (Total)"].index[0]
    parcial_idx = summary[summary["Modelo"] == "BERT-Tiny-PT (Parcial)"].index[0]
    teacher_idx = summary[summary["Modelo"] == "BERTimbau-base"].index[0]

    print("BERT-Tiny-PT (Total) vs BERT-Tiny-PT (Parcial):")
    print(f"  MLM Loss: {summary.loc[total_idx, 'MLM Loss']:.4f} vs {summary.loc[parcial_idx, 'MLM Loss']:.4f}")
    print(f"  Sentiment F1: {summary.loc[total_idx, 'Sentiment F1']:.4f} vs {summary.loc[parcial_idx, 'Sentiment F1']:.4f}")
    print(f"  STS Pearson: {summary.loc[total_idx, 'STS Pearson']:.4f} vs {summary.loc[parcial_idx, 'STS Pearson']:.4f}")
    print(f"  RTE F1: {summary.loc[total_idx, 'RTE F1']:.4f} vs {summary.loc[parcial_idx, 'RTE F1']:.4f}")

    print("\nBERT-Tiny-PT (Total) vs BERTimbau-base (Teacher):")
    print(f"  % Performance MLM: {(summary.loc[teacher_idx, 'MLM Loss'] / summary.loc[total_idx, 'MLM Loss']) * 100:.1f}%")
    print(f"  % Performance Sentiment: {(summary.loc[total_idx, 'Sentiment F1'] / summary.loc[teacher_idx, 'Sentiment F1']) * 100:.1f}%")
    print(f"  % Performance STS: {(summary.loc[total_idx, 'STS Pearson'] / summary.loc[teacher_idx, 'STS Pearson']) * 100:.1f}%")
    print(f"  % Performance RTE: {(summary.loc[total_idx, 'RTE F1'] / summary.loc[teacher_idx, 'RTE F1']) * 100:.1f}%")
    print(f"  Redução de parâmetros: {((summary.loc[teacher_idx, 'Parâmetros (M)'] - summary.loc[total_idx, 'Parâmetros (M)']) / summary.loc[teacher_idx, 'Parâmetros (M)']) * 100:.1f}%")
    print(f"  Redução de tamanho: {((summary.loc[teacher_idx, 'Tamanho (MB)'] - summary.loc[total_idx, 'Tamanho (MB)']) / summary.loc[teacher_idx, 'Tamanho (MB)']) * 100:.1f}%")

    print("\n" + "=" * 80)
    print("VALIDAÇÃO CONCLUÍDA!")
    print("=" * 80)

Device: cuda


VALIDAÇÃO COMPLETA: BERT-Tiny-PT

Carregando dataset brWaC...
Dataset de teste: 353081 exemplos

AVALIAÇÃO 1: MLM LOSS (Masked Language Modeling)

Avaliando BERT-Tiny-PT (Total)...


Some weights of the model checkpoint at Pedro-Rebollo20/BERT-Tiny-Port were not used when initializing BertForMaskedLM: ['proj_teacher.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Map:   0%|          | 0/353081 [00:00<?, ? examples/s]

  MLM Loss: 3.9650
  Tempo: 451.35s

Avaliando BERT-Tiny-PT (Parcial)...


Map:   0%|          | 0/353081 [00:00<?, ? examples/s]

...

  Accuracy: 0.9360
  F1: 0.9359

                Modelo  Accuracy  F1 (macro)  Tempo (s)
  BERT-Tiny-PT (Total)     0.650    0.603640  21.636369
BERT-Tiny-PT (Parcial)     0.636    0.633413  20.719614
        BERT-Tiny (EN)     0.678    0.667464  18.972538
        BERTimbau-base     0.936    0.935950  86.558848

ANÁLISE DE EFICIÊNCIA

Analisando BERT-Tiny-PT (Total)...


Some weights of BertModel were not initialized from the model checkpoint at Pedro-Rebollo20/BERT-Tiny-Port and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at Pedro-Rebollo20/BERT-Tiny-Port and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Parâmetros: 4.29M
  Tamanho: 16.38 MB

Analisando BERT-Tiny-PT (Parcial)...


Some weights of BertModel were not initialized from the model checkpoint at Pedro-Rebollo20/Bert_tiny_pt_parcial and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at Pedro-Rebollo20/Bert_tiny_pt_parcial and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Parâmetros: 4.39M
  Tamanho: 16.74 MB

Analisando BERT-Tiny (EN)...
  Parâmetros: 4.39M
  Tamanho: 16.74 MB

Analisando BERTimbau-base...
  Parâmetros: 108.92M
  Tamanho: 415.52 MB

                Modelo  Parâmetros (M)  Tamanho (MB)
  BERT-Tiny-PT (Total)        4.292736     16.383301
BERT-Tiny-PT (Parcial)        4.385920     16.738770
        BERT-Tiny (EN)        4.385920     16.738770
        BERTimbau-base      108.923136    415.516602


RESUMO FINAL - COMPARAÇÃO ENTRE MODELOS

                Modelo  MLM Loss  Sentiment F1  STS Pearson   RTE F1  Parâmetros (M)  Tamanho (MB)
  BERT-Tiny-PT (Total)  3.964960      0.083333     0.439960 0.603640        4.292736     16.383301
BERT-Tiny-PT (Parcial)  2.154681      0.190476     0.576279 0.633413        4.385920     16.738770
        BERT-Tiny (EN)  5.713826      0.358333     0.526224 0.667464        4.385920     16.738770
        BERTimbau-base  1.757704      0.290196     0.613936 0.935950      108.923136    415.516602

ANÁLISE COMP